# Week 7: QLoRA Fine-Tuning – Outperform Frontier Models on a Specific Task

**PR focus:** This notebook shows *understanding and implementation of QLoRA*: we train an **open-source model** on a **specific task** (clinical note generation from doctor–patient dialogue) and **compare it to a frontier model** (GPT-4 via OpenRouter) to try to **outperform it on that task**.

- **Task:** Given a dialogue, generate a structured clinical note (narrow, well-defined task).
- **QLoRA:** 4-bit quantized base model (NF4, double quant) + LoRA adapters — keeps memory low while fine-tuning.
- **LoRA:** Low-rank adaptation (e.g. `q_proj`, `v_proj`); only ~0.2% of parameters trained.
- **Outperform frontier:** We evaluate our fine-tuned model vs GPT-4 on the same eval set; on task-specific data a small tuned model can match or beat a generalist frontier model.

**Model:** Default TinyLlama (no sign-up). Optional: [LLaMA 3.2](https://huggingface.co/meta-llama/Llama-3.2-3B) — set `HF_TOKEN` and `MODEL_NAME`.

**What you need to do:**
1. **Run in order:** Run the install cell (Cell 1) once, then use **Kernel → Run All** so every variable is defined.
2. **Model:** Default is TinyLlama (no sign-up). For LLaMA 3.2: get access at the link above, then set env `HF_TOKEN` and change `MODEL_NAME` in the config cell.
3. **Weights & Biases:** Optional. Training runs without it. To log to W&B, set `report_to="wandb"` in the training args and run `wandb.login()` in the next cell.
4. **OpenRouter (GPT-4 comparison):** Optional. Set env `OPENROUTER_API_KEY` only if you want to compare against a frontier model.
5. **GPU:** Training needs a GPU (e.g. Colab T4/A100). On CPU it will be very slow or OOM.

In [ ]:
# Install deps first (run this cell once, then Run All). Restart kernel if peft still missing.
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "torch", "transformers", "accelerate", "peft", "trl", "bitsandbytes", "datasets", "wandb", "gradio", "requests"])

In [ ]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer
from datasets import load_dataset
import wandb
import json
import os
import numpy as np
from tqdm import tqdm
import requests
import gradio as gr

In [ ]:
# --- QLoRA + LoRA config (open-source model, specific task, try to beat frontier) ---
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # or "meta-llama/Llama-3.2-3B" + HF_TOKEN
HF_TOKEN = os.getenv("HF_TOKEN")

# Task: clinical note generation from dialogue (specific task for comparison vs GPT-4)
DATASET_NAME = "ClinicianFOCUS/ACI-Bench-Refined"
OUTPUT_DIR = "./lora-medical-llama"
WANDB_PROJECT = "qlora-medical-extraction"

# LoRA: low-rank adapters on attention projections (small trainable set)
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
TARGET_MODULES = ["q_proj", "v_proj"]

# Training
LEARNING_RATE = 2e-4
BATCH_SIZE = 4
GRAD_ACC_STEPS = 4
EPOCHS = 3
MAX_SEQ_LENGTH = 512
WARMUP_STEPS = 100
LOGGING_STEPS = 10
EVAL_STEPS = 200
SAVE_STEPS = 200

# QLoRA: 4-bit quantization (NF4 + double quant) so base model fits in low memory
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# OpenRouter (optional – only needed for frontier-model comparison later)
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

In [ ]:
dataset = load_dataset(DATASET_NAME, token=HF_TOKEN)
train_dataset = dataset["train"]

# Create validation split (90/10)
split_dataset = train_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

def format_instruction(example):
    instruction = f"Generate a structured clinical note from the following doctor-patient dialogue:\n{example['dialogue']}"
    response = example['note']
    return f"### Instruction:\n{instruction}\n\n### Response:\n{response}"

train_dataset = train_dataset.map(lambda x: {"text": format_instruction(x)})
eval_dataset = eval_dataset.map(lambda x: {"text": format_instruction(x)})

print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# On MPS (Apple Silicon), 4-bit + 8-bit optimizer are not supported → use bf16 + LoRA (no QLoRA)
_use_cuda = torch.cuda.is_available()
_is_mps = getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available()
_use_qlora = _use_cuda  # QLoRA only on CUDA; on MPS use full bf16 + LoRA so training runs

if _use_qlora:
    # QLoRA: 4-bit base + LoRA (CUDA)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        token=HF_TOKEN,
    )
    model = prepare_model_for_kbit_training(model)
else:
    # LoRA only: bf16 base (MPS/CPU – avoids bitsandbytes 8-bit optimizer)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
        token=HF_TOKEN,
    )

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
if _is_mps:
    print("(MPS: using bf16 + LoRA; run on Colab/CUDA for QLoRA.)")

In [ ]:
# MPS: must use adamw_torch (bitsandbytes 8-bit not implemented); CUDA: can use paged_adamw_8bit
_optim = "paged_adamw_8bit" if _use_cuda else "adamw_torch"
_dataloader_pin_memory = _use_cuda

# SFTConfig (TRL) includes training args + SFT-specific: dataset_text_field, max_length
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACC_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    num_train_epochs=EPOCHS,
    logging_steps=LOGGING_STEPS,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=True,
    optim=_optim,
    dataloader_pin_memory=_dataloader_pin_memory,
    report_to="none",
    run_name="medical-llama-qlora",
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
)

In [ ]:
# Optional: if you set report_to="wandb" above, run wandb.login() and paste your API key when prompted
# wandb.login()

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

In [ ]:
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
# wandb.finish()  # only if you used report_to="wandb"

In [ ]:
def evaluate_model(model, tokenizer, eval_dataset, num_samples=50):
    model.eval()
    correct = 0
    total = 0
    for example in tqdm(eval_dataset.select(range(min(num_samples, len(eval_dataset))))):
        prompt = example["text"].split("### Response:\n")[0] + "### Response:\n"
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.1)
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        if "### Response:\n" in response:
            pred = response.split("### Response:\n")[-1].strip()
        else:
            pred = ""
        ref = example["text"].split("### Response:\n")[-1].strip()
        # Simple heuristic: check if note-like (you can improve)
        if "note" in pred.lower() and len(pred) > 50:
            correct += 1
        total += 1
    return correct / total if total > 0 else 0

accuracy = evaluate_model(model, tokenizer, eval_dataset)
print(f"Validation accuracy: {accuracy:.4f}")

In [ ]:
def query_openrouter(prompt, model="openai/gpt-4"):
    response = requests.post(
        url="https://openrouter.ai/api/v1/chat/completions",
        headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}"},
        json={
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.1,
            "max_tokens": 512,
        }
    )
    return response.json()["choices"][0]["message"]["content"]

subset_size = 20
eval_subset = eval_dataset.select(range(min(subset_size, len(eval_dataset))))
gpt4_accuracy = None
if OPENROUTER_API_KEY:
    gpt4_correct = 0
    for example in tqdm(eval_subset):
        prompt = example["text"].split("### Response:\n")[0] + "### Response:\n"
        try:
            gpt4_pred = query_openrouter(prompt, model="openai/gpt-4")
        except Exception:
            gpt4_pred = ""
        if "note" in gpt4_pred.lower() and len(gpt4_pred) > 50:
            gpt4_correct += 1
    gpt4_accuracy = gpt4_correct / subset_size
    print(f"GPT-4 accuracy: {gpt4_accuracy:.4f}")
print(f"Your QLoRA (fine-tuned open-source) accuracy: {accuracy:.4f}")
if gpt4_accuracy is not None:
    print(f"→ Outperform frontier? {'Yes' if accuracy >= gpt4_accuracy else 'Close / No'} (vs GPT-4 {gpt4_accuracy:.4f})")

In [ ]:
from peft import PeftModel

# Reload base model + LoRA weights for inference
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
    token=HF_TOKEN,
)
tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
model.eval()

def generate_response(dialogue, temperature=0.1, max_new_tokens=256):
    prompt = f"### Instruction:\nGenerate a structured clinical note from the following doctor-patient dialogue:\n{dialogue}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "### Response:\n" in response:
        response = response.split("### Response:\n")[-1].strip()
    return response

iface = gr.Interface(
    fn=generate_response,
    inputs=[
        gr.Textbox(lines=5, placeholder="Enter doctor-patient dialogue...", label="Dialogue"),
        gr.Slider(0.0, 1.0, value=0.1, label="Temperature"),
        gr.Slider(64, 512, value=256, step=32, label="Max New Tokens")
    ],
    outputs=gr.Textbox(lines=10, label="Generated Clinical Note"),
    title="Medical Note Generator (QLoRA fine-tuned open-source model)",
    description="Enter a dialogue between doctor and patient to generate a structured clinical note."
)

# Headless / no display: share=True gives a public URL; inbrowser=False skips opening a browser
iface.launch(share=True, inbrowser=False)
# Open the printed URL (e.g. https://xxx.gradio.live) in any browser on any device.